# Exercícios — Pipeline completo

Um projeto de ponta a ponta com dados de tipos mistos e valores faltantes, sem vazamento. Soluções em `# @title`.

In [ ]:
# bibliotecas base
import numpy as np
import pandas as pd

# Plotly para os gráficos (interativos e leves no Colab)
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

# paleta do curso (a mesma do site)
AZUL, VERMELHO, VERDE = "#3266ad", "#c0392b", "#1a7a4a"
TINTA, SUAVE = "#1c1e15", "#6b7050"

# reprodutibilidade: uma única semente para tudo que é aleatório
SEMENTE = 42
np.random.seed(SEMENTE)

## Preparação — um conjunto realista (numérico + categórico + faltantes)

In [ ]:
# @title Solução
rng = np.random.RandomState(SEMENTE)
n = 500
idade = rng.normal(50, 12, n)
pressao = rng.normal(125, 16, n)
grupo = rng.choice(["A", "B", "C"], n)
logito = 0.04*(idade-50) + 0.03*(pressao-125) + (grupo=="C")*1.0 + rng.normal(0, 0.6, n)
y = (logito > np.median(logito)).astype(int)
# injeta valores faltantes nas numericas
idade[rng.rand(n) < 0.12] = np.nan
pressao[rng.rand(n) < 0.08] = np.nan
df = pd.DataFrame({"idade": idade, "pressao": pressao, "grupo": grupo})
print("faltantes por coluna:\n", df.isna().sum())

## Exercício 1 — Montar o ColumnTransformer

In [ ]:
# @title Solução
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

num = ["idade", "pressao"]
cat = ["grupo"]
prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="mean")), ("esc", StandardScaler())]), num),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder())]), cat),
])
modelo = Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=5000))])
print("pipeline montado. A imputacao fica DENTRO, ajustada so no treino de cada dobra.")

## Exercício 2 — Comparar modelos com o mesmo pipeline

In [ ]:
# @title Solução
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score

for nome, clf in [("logistica", LogisticRegression(max_iter=5000)),
                  ("random forest", RandomForestClassifier(n_estimators=200, random_state=SEMENTE)),
                  ("grad. boosting", GradientBoostingClassifier(random_state=SEMENTE))]:
    pipe = Pipeline([("prep", prep), ("clf", clf)])
    ac = cross_val_score(pipe, df, y, cv=5).mean()
    print(nome.ljust(15), "acuracia CV:", round(ac, 3))

## Exercício 3 — Estimativa final honesta

In [ ]:
# @title Solução
from sklearn.model_selection import train_test_split, GridSearchCV

df_tr, df_te, y_tr, y_te = train_test_split(df, y, test_size=0.25, random_state=SEMENTE, stratify=y)
pipe = Pipeline([("prep", prep), ("clf", RandomForestClassifier(random_state=SEMENTE))])
grade = {"clf__n_estimators": [100, 300], "clf__max_depth": [None, 5]}
busca = GridSearchCV(pipe, grade, cv=5).fit(df_tr, y_tr)
print("melhor CV (otimista):", round(busca.best_score_, 3))
print("acuracia no TESTE separado (honesta):", round(busca.score(df_te, y_te), 3))

## Exercício 4 — Reprodutibilidade

In [ ]:
# @title Solução
import joblib
joblib.dump(busca.best_estimator_, "modelo_final.joblib")
recarregado = joblib.load("modelo_final.joblib")
print("modelo salvo e recarregado; previsoes identicas:",
      np.array_equal(busca.best_estimator_.predict(df_te), recarregado.predict(df_te)))
print("semente fixa + random_state em tudo + modelo salvo = resultado verificavel e reusavel.")